
### 01 · Bronze: incremental trip ingestion with Auto Loader (Phase 1)

## 1. Parameters

In [0]:
catalog = dbutils.widgets.get("catalog")
service = dbutils.widgets.get("service")
schema_hints = dbutils.widgets.get("schema_hints")
reset = dbutils.widgets.get("reset")

print(catalog, service, schema_hints, reset)

In [0]:
CATALOG = dbutils.widgets.get("catalog")
SERVICE = dbutils.widgets.get("service")
SCHEMA_HINTS = dbutils.widgets.get("schema_hints").strip()

SOURCE_PATH = f"/Volumes/{CATALOG}/landing/raw/{SERVICE}/"
STATE_DIR = f"/Volumes/{CATALOG}/ops/checkpoints/bronze/{SERVICE}"
CHECKPOINT = f"{STATE_DIR}/checkpoint"
SCHEMA_LOC = f"{STATE_DIR}/schema"
TARGET = f"{CATALOG}.bronze.{SERVICE}_trips_raw"
LOG_TABLE = f"{CATALOG}.ops.ingestion_log"

In [0]:
print(f"source:     {SOURCE_PATH}")
print(f"target:     {TARGET}")
print(f"checkpoint: {CHECKPOINT}")
print(f"schema:     {SCHEMA_LOC}")

### 2. Reset

In [0]:
if dbutils.widgets.get("reset") == "yes":
    spark.sql(f"DROP TABLE IF EXISTS {TARGET}")
    dbutils.fs.rm(STATE_DIR, True)
    if spark.catalog.tableExists(LOG_TABLE):
        spark.sql(f"DELETE FROM {LOG_TABLE} WHERE layer = 'bronze' AND service = '{SERVICE}'")
    dbutils.notebook.exit(f"Reset done for {SERVICE}. Set reset = no and run again.")


### 3. Run log table

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
  run_id         STRING    COMMENT 'Unique ID of the ingestion run',
  layer          STRING    COMMENT 'bronze / silver / gold',
  service        STRING    COMMENT 'yellow / green',
  status         STRING    COMMENT 'success, success_after_schema_evolution, failed',
  started_at     TIMESTAMP,
  finished_at    TIMESTAMP,
  duration_s     DOUBLE,
  rows_ingested  BIGINT,
  files_ingested BIGINT,
  source_months  STRING    COMMENT 'Comma-separated YYYY-MM of files loaded in this run',
  note           STRING
)
COMMENT 'One row per ingestion run'
""")


### 4. Stream definition

In [0]:
import datetime
import time
import uuid

from pyspark.sql import functions as F

In [0]:
def build_stream(run_id: str):
    reader = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", SCHEMA_LOC)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    )
    if SCHEMA_HINTS:
        reader = reader.option("cloudFiles.schemaHints", SCHEMA_HINTS)

    return reader.load(SOURCE_PATH).select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
        F.col("_metadata.file_modification_time").alias("_source_file_mtime"),
        F.regexp_extract(F.col("_metadata.file_path"), r"(\d{4}-\d{2})\.parquet$", 1).alias("_source_month"),
        F.lit(SERVICE).alias("_service_type"),
        F.lit(run_id).alias("_ingest_run_id"),
        F.current_timestamp().alias("_ingested_at"),
    )

In [0]:
def run_stream_once(run_id: str) -> None:
    query = (
        build_stream(run_id)
        .writeStream
        .option("checkpointLocation", CHECKPOINT)
        .option("mergeSchema", "true")# let the Delta table accept new columns
        .trigger(availableNow=True)# process all new files, then stop
        .toTable(TARGET)
    )
    query.awaitTermination()


In [0]:
def write_log(run_id, status, started_at, duration_s, rows, files, months, note):
    row = [(run_id, "bronze", SERVICE, status, started_at, datetime.datetime.now(datetime.timezone.utc),
            float(duration_s), int(rows), int(files), months, note)]
    spark.createDataFrame(row, schema=spark.table(LOG_TABLE).schema) \
         .write.mode("append").saveAsTable(LOG_TABLE)

In [0]:
run_id = str(uuid.uuid4())
started_at = datetime.datetime.now(datetime.timezone.utc)
t0 = time.time()
status, note = "success", None

try:
    run_stream_once(run_id)
except Exception as e:
    msg = str(e)
    if "UNKNOWN_FIELD" in msg or "UnknownFieldException" in msg:
        
        print("New column(s) found in source files. Schema updated; restarting the stream once.")
        note = "Schema evolved: " + msg[:400]
        run_stream_once(run_id)
        status = "success_after_schema_evolution"
    else:
        write_log(run_id, "failed", started_at, time.time() - t0, 0, 0, "", msg[:1000])
        raise

In [0]:
duration_s = time.time() - t0
stats = (
    spark.table(TARGET)
    .filter(F.col("_ingest_run_id") == run_id)
    .agg(
        F.count(F.lit(1)).alias("rows"),
        F.countDistinct("_source_file").alias("files"),
        F.concat_ws(",", F.array_sort(F.collect_set("_source_month"))).alias("months"),
    )
    .first()
)



In [0]:
write_log(run_id, status, started_at, duration_s, stats["rows"], stats["files"], stats["months"], note)

print(f"status:   {status}")
print(f"rows:     {stats['rows']:,}")
print(f"files:    {stats['files']}  months: {stats['months'] or '-'}")
print(f"duration: {duration_s:.1f} s")
if stats["rows"]:
    print(f"rate:     {stats['rows'] / duration_s:,.0f} rows/s")

In [0]:
display(spark.table(LOG_TABLE)
    .filter((F.col("layer") == "bronze") & (F.col("service") == SERVICE))
    .orderBy(F.desc("started_at"))
)

In [0]:
display(spark.table(TARGET)
    .groupBy("_source_month")
    .agg(F.count(F.lit(1)).alias("rows"), F.countDistinct("_source_file").alias("files"))
    .orderBy("_source_month")
)

In [0]:
bronze = spark.table(TARGET)
if "_rescued_data" in bronze.columns:
    rescued = bronze.filter(F.col("_rescued_data").isNotNull())
    display(rescued.groupBy("_source_month").count().orderBy("_source_month"))
    display(rescued.select("_source_month", "_rescued_data").limit(10))
else:
    print("No _rescued_data column (schema was provided, not inferred).")

In [0]:
if "cbd_congestion_fee" in bronze.columns:
    display(
        bronze.groupBy("_source_month")
        .agg(
            F.count(F.lit(1)).alias("rows"),
            F.sum(F.col("cbd_congestion_fee").isNull().cast("int")).alias("cbd_fee_null"),
            F.round(F.sum("cbd_congestion_fee"), 2).alias("cbd_fee_total"),
        )
        .orderBy("_source_month")
    )
else:
    print("cbd_congestion_fee not present yet: no 2025 files have been loaded.")

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {TARGET}").select("version", "timestamp", "operation", "operationMetrics").limit(10))

In [0]:
# did Jan 2025 load, and does the table have the new column?
print("Files in source folder:")
for f in dbutils.fs.ls(SOURCE_PATH):
    print("  ", f.name, f.size)

print("\nLatest runs:")
display(spark.table(LOG_TABLE)
        .filter((F.col("layer") == "bronze") & (F.col("service") == SERVICE))
        .orderBy(F.desc("started_at")).limit(3))

cols = [r.col_name for r in spark.sql(f"DESCRIBE TABLE {TARGET}").collect()]
print("cbd_congestion_fee in table:", "cbd_congestion_fee" in cols)

display(spark.sql(f"""
  SELECT _source_month, COUNT(*) AS rows, COUNT(_rescued_data) AS rescued_rows
  FROM {TARGET} GROUP BY _source_month ORDER BY _source_month
"""))

In [0]:
# Which run loaded 2025-01, and was it logged?
display(spark.sql(f"""
  SELECT _ingest_run_id, COUNT(*) AS rows, COUNT(DISTINCT _source_file) AS files,
         MIN(_ingested_at) AS first_ts, MAX(_ingested_at) AS last_ts
  FROM {TARGET} WHERE _source_month = '2025-01' GROUP BY _ingest_run_id
"""))

display(spark.sql(f"SELECT * FROM {LOG_TABLE} WHERE service = '{SERVICE}' ORDER BY started_at DESC"))

display(spark.sql(f"DESCRIBE HISTORY {TARGET}")
        .select("version", "timestamp", "operation", "operationParameters").limit(5))

In [0]:
display(spark.sql(f"""
  SELECT l.service, l.logged_rows, t.table_rows, l.logged_rows = t.table_rows AS reconciled
  FROM (SELECT service, SUM(rows_ingested) AS logged_rows
        FROM {LOG_TABLE} WHERE layer = 'bronze' GROUP BY service) l
  JOIN (SELECT 'yellow' AS service, COUNT(*) AS table_rows FROM {CATALOG}.bronze.yellow_trips_raw
        UNION ALL
        SELECT 'green', COUNT(*) FROM {CATALOG}.bronze.green_trips_raw) t
  ON l.service = t.service
"""))